Summary: ML Gold Features Table (ml_listing_features)

Purpose
This notebook creates a feature-rich, listing-level Gold table designed for machine learning tasks such as clustering, segmentation, or predictive modeling. It consolidates cleaned data from the Silver layer and engineers new features that reflect listing performance, host behavior, and market dynamics. The output is saved as a Parquet file for efficient reuse in modeling workflows and dashboards.

What It Does

Centralized Setup
- Loads cleaned Silver tables (listings, calendar, reviews) using pathlib for reproducibility.
- Creates the data/gold directory if it doesn’t exist.

Calendar Aggregation (Future-Oriented)
- Computes listing-level metrics from the calendar table:
- future_calendar_days, future_available_days, future_avg_price
- Derives future_occupancy_rate as a proxy for booking intent
- Focuses on forward-looking availability, not historical bookings

Review Aggregation
- Calculates:
- review_count, first_review_date, last_review_date
- reviews_per_month based on review span
- Captures listing maturity and engagement velocity

Listing Feature Selection & Cleaning
- Selects relevant listing attributes (e.g. price, room type, host status)
- Converts host_is_superhost to binary flag
- Fills missing review_scores_rating with 0 (neutral baseline)

Feature Consolidation
- Merges listings with calendar and review aggregates
- Drops redundant listing_id column
- Fills missing review metrics with 0 to support modeling

Beds Imputation
- Applies a three-tiered strategy to fill missing or zero beds values:
- Rule 1: If accommodates == 1, set beds = 1
- Rule 2: If accommodates == bedrooms, set beds = accommodates
- Rule 3: Use contextual median by neighbourhood_cleansed and room_type

Revenue Imputation
- Fills missing estimated_revenue_l365d using:
- future_avg_price × estimated_occupancy_l365d
- Ensures all listings have a revenue estimate for clustering or ranking

Output
- Saves the final ml_listing_features table as a Parquet file in the Gold layer
- Ready for use in clustering notebooks, dashboards, or model training


Failsafes Built In
- Path resolution: Uses pathlib for OS-safe, reproducible directory setup
- Null handling: Fills or imputes missing values with principled logic
- Contextual imputation: Uses domain-aware medians for beds
- Print statements: Confirms row counts and successful saves
- Parquet output: Ensures fast, schema-preserving storage for downstream use


Alignment with Scalable, Reproducible Pipelin
- Modularity - Each transformation is isolated and clearly labelled
- Reproducibility - Uses deterministic logic and fixed paths for consistent outputs
- Auditability - Includes print statements and preserves intermediate logic (e.g. imputation rules)
- Portability - pathlib ensures compatibility across environments
- Scalability - Easily extendable to new features or modeling tasks
- Business Relevance - Features reflect host behavior, listing performance, and market dynamics
- Efficiency - Parquet format enables fast reads and compact storag


In [1]:
# ---------------------------------------------------------
# ML GOLD SETUP: Imports & Path Resolution
# Purpose: Centralize setup for ML feature engineering.
# ---------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
silver_dir = project_root / "data" / "silver"
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Load Silver tables
df_listings = pd.read_parquet(silver_dir / "listings_clean.parquet")
df_calendar = pd.read_parquet(silver_dir / "calendar_clean.parquet")
df_reviews = pd.read_parquet(silver_dir / "reviews_clean.parquet")

print("✅ Silver tables loaded")

✅ Silver tables loaded


In [2]:
# ---------------------------------------------------------
# ML GOLD TABLE: ml_listing_features
# Purpose: Feature-rich listing-level table for clustering.
# ---------------------------------------------------------

# Calendar Aggregates (Future-Oriented)
calendar_agg = (
    df_calendar
    .groupby("listing_id")
    .agg(
        future_calendar_days=("date", "count"),
        future_available_days=("available", "sum"),
        future_avg_price=("price", "mean")
    )
    .assign(
        future_occupancy_rate=lambda df: 1 - (df["future_available_days"] / df["future_calendar_days"])
    )
    .reset_index()
)

print(f"✅ Calendar metrics aggregated for {len(calendar_agg):,} listings")

# Review Aggregates
review_agg = (
    df_reviews
    .groupby("listing_id")
    .agg(
        review_count=("id", "count"),
        first_review_date=("date", "min"),
        last_review_date=("date", "max")
    )
    .assign(
        review_span_days=lambda df: (df["last_review_date"] - df["first_review_date"]).dt.days,
        reviews_per_month=lambda df: df["review_count"] / (df["review_span_days"] / 30).replace(0, pd.NA)
    )
    .drop(columns=["review_span_days"])
    .reset_index()
)

print(f"✅ Review metrics aggregated for {len(review_agg):,} listings")

# Listing Selection
listing_cols = [
    "id", "host_id", "room_type", "neighbourhood_cleansed", "price",
    "host_is_superhost", "host_response_rate", "review_scores_rating",
    "beds", "accommodates", "minimum_nights", "maximum_nights",
    "estimated_revenue_l365d", "estimated_occupancy_l365d",
    "number_of_reviews_ltm", "number_of_reviews"
]
df_listings = df_listings[listing_cols]

# Superhost Treatment
df_listings["host_is_superhost"] = df_listings["host_is_superhost"].fillna("f").map({"t": 1, "f": 0})

# Review Score Treatment
df_listings["review_scores_rating"] = df_listings["review_scores_rating"].fillna(0)

# Merge All Sources
ml_listing_features = (
    df_listings
    .merge(calendar_agg, how="left", left_on="id", right_on="listing_id")
    .merge(review_agg, how="left", on="listing_id")
    .drop(columns=["listing_id"])
)

# Review Count & Velocity Treatment
ml_listing_features["review_count"] = ml_listing_features["review_count"].fillna(0)
ml_listing_features["reviews_per_month"] = ml_listing_features["reviews_per_month"].fillna(0)

print(f"✅ ML listing features assembled: {len(ml_listing_features):,} rows")

✅ Calendar metrics aggregated for 94,554 listings
✅ Review metrics aggregated for 70,316 listings
✅ ML listing features assembled: 94,559 rows


C:\Users\emand\AppData\Local\Temp\ipykernel_29884\2276868797.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ml_listing_features["reviews_per_month"] = ml_listing_features["reviews_per_month"].fillna(0)


In [3]:
# ---------------------------------------------------------
# BED IMPUTATION LOGIC
# Purpose: Improve missing or zero bed values using rules + context.
# ---------------------------------------------------------

ml_listing_features["beds_imputed"] = ml_listing_features["beds"].copy()

# Rule 1: If accommodates == 1 → beds = 1
ml_listing_features.loc[ml_listing_features["accommodates"] == 1, "beds_imputed"] = 1

# Rule 2: If accommodates == bedrooms → beds = accommodates
if "bedrooms" in df_listings.columns:
    mask = ml_listing_features["accommodates"] == df_listings["bedrooms"]
    ml_listing_features.loc[mask, "beds_imputed"] = ml_listing_features.loc[mask, "accommodates"]

# Rule 3: Contextual median by neighbourhood + room_type
group_median = (
    ml_listing_features[ml_listing_features["beds_imputed"].notna() & (ml_listing_features["beds_imputed"] > 0)]
    .groupby(["neighbourhood_cleansed", "room_type"])["beds_imputed"]
    .median()
)

def impute_beds(row):
    if pd.isna(row["beds_imputed"]) or row["beds_imputed"] == 0:
        return group_median.get((row["neighbourhood_cleansed"], row["room_type"]), np.nan)
    return row["beds_imputed"]

ml_listing_features["beds_imputed"] = ml_listing_features.apply(impute_beds, axis=1)

print("✅ Beds imputed using rules and contextual medians")

# ---------------------------------------------------------
# REVENUE IMPUTATION LOGIC
# Purpose: Fill missing estimated revenue using future price × occupancy.
# ---------------------------------------------------------

ml_listing_features["estimated_revenue_l365d"] = ml_listing_features["estimated_revenue_l365d"].fillna(
    ml_listing_features["future_avg_price"] * ml_listing_features["estimated_occupancy_l365d"]
)

print("✅ Revenue imputed using future price × occupancy")

C:\Users\emand\AppData\Local\Temp\ipykernel_29884\2660355229.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["neighbourhood_cleansed", "room_type"])["beds_imputed"]


✅ Beds imputed using rules and contextual medians
✅ Revenue imputed using future price × occupancy


In [4]:
# ---------------------------------------------------------
# SAVE GOLD TABLE
# Purpose: Persist ML-ready listing features for clustering.
# ---------------------------------------------------------

output_path = gold_dir / "ml_listing_features.parquet"
ml_listing_features.to_parquet(output_path, index=False)

print(f"✅ Saved ML-ready listing features: {len(ml_listing_features):,} rows → {output_path.name}")

✅ Saved ML-ready listing features: 94,559 rows → ml_listing_features.parquet
